In [ ]:
REPO_URL = "https://github.com/30nikamvaishnavi/autonomous-driving-perception-stack.git"
BRANCH = "main"
DATA_DIR = None  # Override to use a different runtime dataset path.
OUT_DIR = None  # Override to change where outputs and checkpoints are saved.
WANDB_PROJECT = "adpstack"
MODEL_PATH = None  # Optional .pt path for export/benchmark notebooks.


In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

SMOKE_STARTED_AT = time.monotonic()
IS_COLAB = (not Path("/kaggle").exists()) and (Path("/content").exists() or "COLAB_RELEASE_TAG" in os.environ)
IS_KAGGLE = Path("/kaggle").exists()
if IS_COLAB == IS_KAGGLE:
    raise RuntimeError("Unsupported or ambiguous runtime: use Google Colab or Kaggle.")

if IS_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    runtime_root = Path("/content")
    data_default = Path("/content/drive/MyDrive/adpstack/data/bdd100k")
    output_default = Path("/content/drive/MyDrive/adpstack/outputs")
else:
    runtime_root = Path("/kaggle/working")
    data_default = Path("/kaggle/input")
    output_default = Path("/kaggle/working/outputs")

DATA_DIR = Path(DATA_DIR) if DATA_DIR is not None else data_default
OUT_DIR = Path(OUT_DIR) if OUT_DIR is not None else output_default
OUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ["WANDB_PROJECT"] = WANDB_PROJECT

REPO_DIR = runtime_root / "autonomous-driving-perception-stack"
if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
elif REPO_DIR.exists():
    raise FileExistsError(f"{REPO_DIR} exists but is not a Git checkout; remove/rename it and retry.")
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[ml,export]"],
    cwd=REPO_DIR,
    check=True,
)
import torch

print(f"Runtime: {'Colab' if IS_COLAB else 'Kaggle'}")
print(f"Repository: {REPO_DIR}")
print(f"Data: {DATA_DIR}")
print(f"Outputs: {OUT_DIR}")
print(f"PyTorch: {torch.__version__}")
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU is attached. Select a T4 GPU runtime in Colab or enable a GPU accelerator in Kaggle, then restart."
    )
print(f"GPU: {torch.cuda.get_device_name(0)}")


# 01 - BDD100K exploratory data analysis
Run locally or on Colab after downloading BDD100K (see docs/setup.md).

In [ ]:
import json
from pathlib import Path

from adpstack.data.eda import compute_stats, imbalance_report, size_buckets, plot_all

def find_bdd_root(base: Path, max_depth: int = 5) -> Path:
    for labels_file in sorted(base.rglob("det_train.json")):
        relative = labels_file.relative_to(base)
        if len(relative.parts) > max_depth + 3:
            continue
        if relative.parts[-3:-1] != ("labels", "det_20"):
            continue
        candidate = labels_file.parents[2]
        if (candidate / "images/train").is_dir():
            return candidate
    raise FileNotFoundError(
        f"No BDD100K root within {max_depth} levels under {base}; expected labels/det_20/det_train.json and images/train."
    )


BDD_ROOT = find_bdd_root(DATA_DIR)
print(f"BDD100K root: {BDD_ROOT}")
LABELS = BDD_ROOT / "labels" / "det_20" / "det_train.json"
with LABELS.open(encoding="utf-8") as label_file:
    frames = json.load(label_file)
st = compute_stats(frames)
print(st["n_frames"], "frames", st["total_boxes"], "boxes")

In [ ]:
for split in ("train", "val"):
    image_count = len(list((BDD_ROOT / "images" / split).glob("*.jpg")))
    labels_file = BDD_ROOT / "labels" / "det_20" / f"det_{split}.json"
    label_file_count = int(labels_file.is_file())
    print(f"{split}: {image_count} images, {label_file_count} label JSON files")
    if image_count == 0 or label_file_count == 0:
        raise RuntimeError(f"{split} images or labels are missing under {BDD_ROOT}.")

In [ ]:
rep = imbalance_report(st['class_counts'])
print('imbalance ratio:', round(rep['imbalance_ratio'],1))
print(rep['class_weights'])
print(size_buckets(st['areas']))

In [ ]:
paths = plot_all(st, OUT_DIR / "figures")
from IPython.display import Image, display
[display(Image(str(path))) for path in paths]

## Findings (fill in from your run)
- Dominant class: ...
- Rarest class + imbalance ratio: ...
- Share of small objects (<32px): ...
- Night / rainy share: ...

**Mitigations:** mosaic + mixup augmentation, inverse-frequency class weights, oversample night/rain frames, higher imgsz for small objects.